# BiPert-ID: perturbation identification with a training-free linear prototype model

This notebook recomputes, from a compact data bundle, every perturbation-identification number of the paper
*How Far Can Linear Go? BiPert, a Unified Framework for Single-Cell Perturbation Modelling*
(model in Section 4.2, results in Section 5, cross-task analysis in Section 6, details in the appendix).

**The task in one paragraph.** A *knockdown* switches one gene off in a population of cells. Given a few cells
from a cell line (or donor) that the model has never seen, name the gene that was knocked down, among 1,157
candidates. The benchmark is ExPert's: 22 datasets in 12 folds; each fold holds out a cell line, a T-cell donor or
a T-cell condition, which gives 34 held-out *views* (one dataset scored in one fold).

**What is recomputed here**

1. The full benchmark: BiPert-ID on all 34 views, on ExPert's exact test cells (main-text Table 3), with the
   differences and intervals of Appendix F.
2. The four held-out cell lines of the transfer benchmark: HepG2, Jurkat, K562, RPE1 (main-text Table 2 and
   Table S3).
3. The score ablation (Fig. S2).
4. Which genes are identified: by effect size and by ExPert's gene panel (Tables 4 and 5), and the overlap of the
   two methods.
5. Transfer predicts identification (Table 6) and the logistic regression of Table S2.

Each section ends with a table `paper value | recomputed | difference | OK`; the last cell lists every check.

**What is shipped and not recomputed**

* *ExPert.* Its numbers are computed here from the pseudobulk ranks that ExPert's authors released for their
  model on the same test cells (`expert_pooled.csv.gz`). Running ExPert itself needs a GPU and its checkpoints.
* *Rows marked † in the paper* (GEARS, scGPT, GenePert, own-gene drop, RNA fingerprinting, centroid) are values
  published by ExPert's authors; they are shown as quoted constants.
* *Summary statistics of the training data.* For each training context the bundle holds the effect of each
  candidate knockdown and three per-gene vectors computed over all knockdowns of the context (see "Loading the
  bundle"). The test cells are shipped as raw counts.

**Requirements:** numpy, scipy, pandas, matplotlib. **Memory:** about 6 GB. **Runtime:** printed in the last
cell; `FAST = True` runs only the four cell lines of Table 2 (sections 2 to 5).

## Configuration

In [ ]:
from pathlib import Path

DATA_DIR = Path("./identification_data")
# Google Colab: copy the identification_data folder to a folder BiPert on your Drive, then uncomment the next two lines.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_DIR = Path("/content/drive/MyDrive/BiPert/identification_data")

FAST = False          # True: only the four cell lines of Table 2; the full-benchmark checks are listed as SKIPPED
TOL = 0.005           # a check passes if |recomputed - paper| <= TOL (the paper prints 2 or 3 decimals)
N_BOOT = 2000         # bootstrap resamples

In [ ]:
import json, time
from functools import reduce

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp

pd.set_option("display.width", 220); pd.set_option("display.max_rows", 200)
T_START = time.time()

## Bookkeeping for the checks

`check` compares one recomputed number with the paper and records PASS or FAIL. Nothing is raised on a failure,
so the notebook always runs to the summary at the end.

In [ ]:
CHECKS = []

def check(section, claim, paper, recomputed, tol=TOL):
    ok = abs(recomputed - paper) <= tol + 1e-9
    CHECKS.append(dict(section=section, claim=claim, paper=paper, recomputed=round(float(recomputed), 4),
                       difference=round(float(recomputed - paper), 4), tolerance=tol, OK="PASS" if ok else "FAIL"))

def check_true(section, claim, condition, detail=""):
    CHECKS.append(dict(section=section, claim=claim, paper="true", recomputed=detail, difference="", tolerance="",
                       OK="PASS" if condition else "FAIL"))

def skipped(section, claim):
    CHECKS.append(dict(section=section, claim=claim, paper="", recomputed="", difference="", tolerance="", OK="SKIPPED (FAST)"))

def show(section):
    display(pd.DataFrame([c for c in CHECKS if c["section"] == section]).drop(columns="section"))

## Loading the bundle

* `benchmark.json`: the 12 folds, their held-out views, and the training contexts of each fold. A *context* is
  the unit over which prototypes are averaged: one per cell line, and primary T cells split by condition (resting,
  stimulated 8 h, stimulated 48 h). Datasets of one context (the two K562 screens; the donors of a T-cell
  condition) are pooled. When a fold holds out a T-cell donor, the T-cell contexts are rebuilt without that donor,
  which is why some contexts have several files (`train_T_Rest_noD2.npz` = resting T cells without donor 2).
* `train_<context>.npz`: summary statistics of one training context.
  `effect` = for each candidate knockdown, mean log-normalised expression of its cells minus that of control
  cells (one row per knockdown, one column per gene). `mad`, `mean_effect` = per gene, robust spread and mean of
  the effect over **all** knockdowns of the context (2,000 to 18,000 of them, not only the candidates).
  `ctrl_profile` = log-normalised mean expression of the control cells.
* `query_<dataset>.npz`: ExPert's test cells of one held-out dataset as raw counts, each cell's total count, its
  true knocked-down gene (used only to score the ranking), and the mean / standard deviation / pooled profile of
  the dataset's unlabelled control cells.
* `expert_pooled.csv.gz`: ExPert's released pseudobulk rank of the true gene for every knockdown of every view.
* `reference_ranks.csv.gz`: the ranks of the authors' own cluster run, used below only to confirm that this
  notebook reproduces it.

Log-normalised means log(1 + 10,000 x count / total count of the cell).

In [ ]:
BENCH = json.load(open(DATA_DIR / "benchmark.json"))
CANDIDATES = np.array(open(DATA_DIR / "candidates.txt").read().split())
PANEL = set(open(DATA_DIR / "expert_input_panel.txt").read().split())
EXPERT = pd.read_csv(DATA_DIR / "expert_pooled.csv.gz").set_index(["fold", "dataset", "label"])["rank"]
REFERENCE = pd.read_csv(DATA_DIR / "reference_ranks.csv.gz").set_index(["fold", "dataset", "label"])
VIEWS = pd.DataFrame([dict(fold=int(k), **v) for k, f in BENCH["folds"].items() for v in f["views"]])
FOUR_LINES = {"ReplogleWeissman2025_HepG2": "HepG2", "ReplogleWeissman2025_Jurkat": "Jurkat",
              "ReplogleWeissman2022_K562_essential": "K562", "ReplogleWeissman2022_rpe1": "RPE1"}
VIEWS["line"] = VIEWS["dataset"].map(FOUR_LINES)
print(f"{len(CANDIDATES):,d} candidates | {len(PANEL):,d} genes in ExPert's panel | {len(VIEWS)} views in {VIEWS.fold.nunique()} folds")
display(VIEWS.groupby("kind").size().rename("views").to_frame().T)

_TRAIN = {}
def load_train(name):
    if name not in _TRAIN:
        z = np.load(DATA_DIR / f"train_{name}.npz")
        _TRAIN[name] = {k: z[k] for k in ("genes", "labels", "effect", "mad", "mean_effect", "ctrl_profile")}
    return _TRAIN[name]

def load_query(dataset):
    z = np.load(DATA_DIR / f"query_{dataset}.npz")
    counts = sp.csr_matrix((z["counts_data"].astype(np.float64), z["counts_indices"].astype(np.int32), z["counts_indptr"]),
                           shape=tuple(z["counts_shape"]))
    q = {k: z[k] for k in z.files if not k.startswith("counts_")}
    q["counts"] = counts; q["sd_fill"] = float(q["sd_fill"])
    assert (np.diff(q["genes"]) > 0).all()
    return q

## The model

BiPert-ID has three steps (Section 4.2 of the paper, details in Appendix B). Nothing is trained.

### Step 1: a prototype for each perturbation

In each training context, the effects are divided gene by gene by their robust spread, the *shared response*
(the direction of the mean effect over all knockdowns) is projected out, and each residual is scaled to unit
length. The prototype of a knockdown is the weighted average of its residuals over the training contexts that
measure it.

In [ ]:
def context_residuals(train, G):
    """Effects E and unit-length specific residuals R of one training context, on the gene set G."""
    sel = np.searchsorted(train["genes"], G)
    E = train["effect"][:, sel].astype(np.float64)
    s = train["mad"][sel]
    s = np.maximum(s, np.quantile(s[s > 0], 0.05))            # floor the spread at its 5th percentile
    Z = E / s
    u = train["mean_effect"][sel] / s; u /= np.linalg.norm(u)   # shared response of this context
    R = Z - np.outer(Z @ u, u)
    R /= np.maximum(np.linalg.norm(R, axis=1, keepdims=True), 1e-12)
    return E, R

**Similarity weights.** A training context counts more when its unperturbed cells resemble those of the
held-out dataset. The control profiles of the held-out dataset and of the training contexts are centred gene by
gene, the cosine of each training context to the held-out dataset is standardised across contexts, and a softmax
turns it into weights. Only unlabelled control cells of the held-out dataset are used. Equal weights give the
*unweighted* version reported in Appendix B.

In [ ]:
def similarity_weights(context_profiles, query_profile):
    P = np.stack([query_profile] + list(context_profiles))
    P = P - P.mean(0, keepdims=True)
    q, C = P[0], P[1:]
    s = (C @ q) / np.maximum(np.linalg.norm(C, axis=1) * np.linalg.norm(q), 1e-12)
    z = (s - s.mean()) / s.std()
    e = np.exp(z - z.max())
    return e / e.sum()

def prototypes(residuals, labels, weights, n_genes):
    """Weighted average of the contexts' residuals. Returns (prototypes, number of contexts per candidate)."""
    acc = np.zeros((len(CANDIDATES), n_genes)); wsum = np.zeros(len(CANDIDATES)); n = np.zeros(len(CANDIDATES))
    for R, lab, w in zip(residuals, labels, weights):
        idx = pd.Index(CANDIDATES).get_indexer(lab)
        acc[idx] += w * R; wsum[idx] += w; n[idx] += 1
    proto = acc / np.where(wsum > 0, wsum, 1.0)[:, None]
    proto /= np.maximum(np.linalg.norm(proto, axis=1, keepdims=True), 1e-12)
    return proto, n

### Step 2: representing a query cell

The cell's log-normalised expression minus the mean of the held-out dataset's control cells, divided by the
standard deviation of those control cells. As in ExPert, the shared response of the query (the direction of the
mean change over the test cells of the view) is projected out.

### Step 3: two signals per candidate gene

* **cosine similarity** between the query and the candidate's prototype, with the candidate's own gene removed
  from both vectors;
* **own-gene drop**: how far the candidate gene's expression lies below the control mean, in control standard
  deviations. It is read from every gene measured in the held-out dataset.

Each signal is standardised across the candidates of a cell and the two are added (Eq. 8 of the paper).
A candidate with no prototype (never measured in training) gets the lowest score.

In [ ]:
def cosine_without_own_gene(A, B, own):
    """Cosine of every row of A (cells) with every row of B (candidates); candidate j's own gene own[j] is removed."""
    num = A @ B.T; na = (A ** 2).sum(1)[:, None]; nb = (B ** 2).sum(1)[None, :]
    col = np.where(own >= 0, own, 0); a = A[:, col]; b = B[np.arange(len(own)), col][None, :]; m = (own >= 0)[None, :]
    num = num - np.where(m, a * b, 0); na = na - np.where(m, a ** 2, 0); nb = nb - np.where(m, b ** 2, 0)
    return num / np.sqrt(np.maximum(na, 1e-12) * np.maximum(nb, 1e-12))

def standardise_rows(S):
    return (S - S.mean(1, keepdims=True)) / np.maximum(S.std(1, keepdims=True), 1e-12)

### Metrics

Scoring follows ExPert's benchmark code. Each cell's scores are turned into log-probabilities (log-softmax), the
log-probabilities of the cells of one knockdown (10 or fewer) are averaged, and the candidates are ranked by this
*pseudobulk* vector. The rank of the true gene counts the candidates scoring above it, plus half of those tied
with it (0 = first). **Top-10 recall** is the share of knockdowns with rank below 10; **MRR** is the mean of
1 / (rank + 1). Scores are kept in single precision and the candidates in ExPert's column order, as in the
original run.

In [ ]:
_ORDER = np.random.default_rng(0).permutation(len(CANDIDATES))      # ExPert's candidate column order

def pseudobulk_rank(scores, labels):
    """scores: cells x candidates (float32, -inf = no prototype). Returns the rank of the true gene per knockdown."""
    L = np.ascontiguousarray(scores[:, _ORDER]); cand = CANDIDATES[_ORDER]
    for i in range(0, len(L), 4096):                             # log-softmax, row by row
        b = L[i:i + 4096]; b -= b.max(1, keepdims=True); s = np.exp(b).sum(1, keepdims=True); b -= np.log(s)
    L = np.where(np.isfinite(L), L, -1e30).astype(np.float64)
    truth = pd.Index(cand).get_indexer(labels)
    out = {}
    for t, rows in pd.Series(np.arange(len(truth))).groupby(truth).indices.items():
        v = L[rows].mean(0)
        out[cand[t]] = (v > v[t]).sum() + 0.5 * ((v == v[t]).sum() - 1)
    return pd.Series(out).sort_index()

def top10(rank):
    return float((rank < 10).mean())

def mrr(rank):
    return float((1 / (rank + 1)).mean())

### One held-out view, start to finish

`run_view` applies the three steps to one view. The gene space is the set of genes measured in every training
dataset of the fold and in the held-out dataset (about 6,200). With `analyses=True` it also returns what the
later sections need: the ablated scores, each knockdown's effect size, and its transfer score.

* **Effect size** of a knockdown: norm of its mean effect over the training contexts that measure it.
* **Transfer score** of a knockdown measured in at least two training contexts: mean, over those contexts, of the
  cosine between its specific residual in one context and the average of its residuals in the others. It uses
  training data only.

In [ ]:
def run_view(fold, dataset, analyses=False):
    f = BENCH["folds"][str(fold)]
    names = sorted(f["training_contexts"])
    train = [load_train(f["training_contexts"][c]) for c in names]
    q = load_query(dataset)
    G = np.intersect1d(reduce(np.intersect1d, [t["genes"] for t in train]), q["genes"])
    qpos = np.searchsorted(q["genes"], G)
    in_G = {s: i for i, s in enumerate(q["symbols"][qpos])}
    in_q = {s: i for i, s in enumerate(q["symbols"])}
    own_G = np.array([in_G.get(p, -1) for p in CANDIDATES])        # the candidate's own gene, in the gene space
    own_q = np.array([in_q.get(p, -1) for p in CANDIDATES])        # ... among all genes of the held-out dataset

    # step 1: prototypes
    E, R = zip(*[context_residuals(t, G) for t in train])
    labels = [t["labels"] for t in train]
    w = similarity_weights([t["ctrl_profile"][np.searchsorted(t["genes"], G)] for t in train], q["ctrl_profile"][qpos])
    proto, n_ctx = prototypes(R, labels, w, len(G))
    proto_eq, _ = prototypes(R, labels, np.ones(len(train)), len(G))
    has = n_ctx > 0

    # step 2: query cells
    rows = q[f"fold{fold}"]; X = q["counts"][rows]; total = q["total"][rows]; truth = q["label"][rows]
    Z = np.log1p(1e4 * X[:, qpos].toarray() / total[:, None])
    sd = q["ctrl_sd"][qpos]; sd = np.maximum(sd, np.quantile(sd[sd > 0], 0.05))
    Z -= q["ctrl_mu"][qpos]; Z /= sd
    drop_G = np.where((own_G >= 0)[None, :], -Z[:, np.where(own_G >= 0, own_G, 0)], 0.0)   # own-gene drop, gene space only
    u = Z.mean(0); u /= np.linalg.norm(u)
    Z -= np.outer(Z @ u, u)                                         # Z is now the query representation

    # step 3: the two signals
    col = np.where(own_q >= 0, own_q, 0)
    sd_t = np.where(own_q >= 0, q["ctrl_sd"][col], q["sd_fill"]); sd_t = np.maximum(sd_t, np.quantile(sd_t[sd_t > 0], 0.05))
    drop = np.where((own_q >= 0)[None, :], -(np.log1p(1e4 * X[:, col].toarray() / total[:, None]) - q["ctrl_mu"][col]) / sd_t, 0.0)

    def final(P):
        raw = cosine_without_own_gene(Z, P, own_G)
        cos = raw.astype(np.float32); cos[:, ~has] = -np.inf
        c0 = np.where(np.isfinite(cos), cos, 0.0).astype(np.float64)
        S = (standardise_rows(c0) + standardise_rows(drop)).astype(np.float32); S[:, ~has] = -np.inf
        return raw, cos, S

    raw, cos, S = final(proto)
    out = pd.DataFrame({"BiPert-ID": pseudobulk_rank(S, truth), "unweighted": pseudobulk_rank(final(proto_eq)[2], truth)})
    out["ExPert"] = EXPERT.loc[(fold, dataset)].reindex(out.index)
    info = dict(fold=fold, dataset=dataset, n_cells=len(rows), n_knockdowns=len(out), n_genes=len(G), n_contexts=len(train),
                top_weights=", ".join(f"{names[i]} {w[i]:.2f}" for i in np.argsort(-w)[:3]))
    if analyses:
        def masked(M):
            M = np.asarray(M, np.float32).copy(); M[:, ~has] = -np.inf; return M
        ci = pd.Index(CANDIDATES).get_indexer(out.index)
        out["cosine only"] = pseudobulk_rank(cos, truth)
        out["own-gene drop only*"] = pseudobulk_rank(masked(np.where((own_G >= 0)[None, :], drop_G, -1e3)), truth)
        out["both*"] = pseudobulk_rank(masked(standardise_rows(raw) + standardise_rows(drop_G)), truth)
        # effect size and transfer score, from the training contexts only
        Esum = np.zeros((len(CANDIDATES), len(G))); A = np.zeros((len(train), len(CANDIDATES), len(G)), np.float32)
        ok = np.zeros((len(train), len(CANDIDATES)), bool)
        for j, (e, r, lab) in enumerate(zip(E, R, labels)):
            idx = pd.Index(CANDIDATES).get_indexer(lab); Esum[idx] += e; A[j, idx] = r; ok[j, idx] = True
        n = ok.sum(0); total_R = A.sum(0); transfer = np.full(len(CANDIDATES), np.nan)
        for p in np.where(n >= 2)[0]:
            vals = []
            for j in np.where(ok[:, p])[0]:
                other = (total_R[p] - A[j, p]) / (n[p] - 1)
                vals.append(A[j, p] @ other / max(np.linalg.norm(other), 1e-12))
            transfer[p] = np.mean(vals)
        out["effect_size"] = np.linalg.norm(Esum / np.maximum(n, 1)[:, None], axis=1)[ci]
        out["transfer"] = transfer[ci]
        out["in_panel"] = [p in PANEL for p in out.index]
    return out, info

### Paired bootstrap

The uncertainty of a difference between two methods comes from which knockdowns happen to be in the benchmark.
Within each view the knockdowns are resampled with replacement, both methods are scored on the same resampled
knockdowns, and the view differences are averaged with equal weight. The 2.5th and 97.5th percentiles of 2,000
resamples give the 95% interval.

In [ ]:
def paired_bootstrap(views, a, b, seed=0):
    """views: list of per-knockdown rank tables. Returns {metric: (difference, low, high)} of method a minus method b."""
    rng = np.random.default_rng(seed); res = {}
    for name, f in (("top10", lambda r: (r < 10).astype(float)), ("mrr", lambda r: 1 / (r + 1))):
        d = [(f(v[a]) - f(v[b])).to_numpy() for v in views]
        boot = np.mean([[x[rng.integers(0, len(x), len(x))].mean() for _ in range(N_BOOT)] for x in d], axis=0)
        res[name] = (float(np.mean([x.mean() for x in d])), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)))
    return res

### Logistic regression

Plain maximum-likelihood logistic regression by Newton's method, with the usual 95% interval (estimate ± 1.96
standard errors from the inverse Hessian) and McFadden's pseudo-R². Written out to avoid a dependency.

In [ ]:
def logistic_fit(X, y, n_iter=100):
    beta = np.zeros(X.shape[1])
    for _ in range(n_iter):
        p = 1 / (1 + np.exp(-X @ beta))
        H = X.T @ (X * (p * (1 - p))[:, None])
        step = np.linalg.solve(H, X.T @ (y - p)); beta += step
        if np.abs(step).max() < 1e-10:
            break
    p = 1 / (1 + np.exp(-X @ beta)); ll = (y * np.log(p) + (1 - y) * np.log(1 - p)).sum()
    se = np.sqrt(np.diag(np.linalg.inv(H)))
    return beta, beta - 1.959964 * se, beta + 1.959964 * se, ll

def z_score(x):
    return (x - x.mean()) / x.std(ddof=1)

## Running the model

All 34 views (or, with `FAST`, the four views of Table 2). The four cell-line views also get the analyses of
sections 3 to 5. The last column confirms that the ranks equal those of the authors' cluster run.

In [ ]:
RUN, INFO = {}, []
todo = VIEWS[VIEWS.line.notna()] if FAST else VIEWS
for v in todo.itertuples():
    t0 = time.time()
    tab, info = run_view(v.fold, v.dataset, analyses=isinstance(v.line, str))
    ref = REFERENCE.loc[(v.fold, v.dataset)].reindex(tab.index)
    info.update(kind=v.kind, top10=top10(tab["BiPert-ID"]), top10_ExPert=top10(tab["ExPert"]),
                same_ranks_as_cluster_run=float(((tab["BiPert-ID"] == ref["rank_bipert_id"]) & (tab["unweighted"] == ref["rank_unweighted"])).mean()),
                seconds=round(time.time() - t0))
    RUN[(v.fold, v.dataset)] = tab; INFO.append(info)
    print(f"fold {v.fold:2d} {v.dataset:38s} {info['n_cells']:6,d} cells {info['n_knockdowns']:5,d} knockdowns | top-10 recall "
          f"BiPert-ID {info['top10']:.3f}, ExPert {info['top10_ExPert']:.3f} | weights: {info['top_weights']} | {info['seconds']} s")
_TRAIN.clear()
INFO = pd.DataFrame(INFO)
SEC = "0 reproduces the cluster run"
check_true(SEC, "every pseudobulk rank equals the authors' cluster run (BiPert-ID and unweighted)",
           bool((INFO.same_ranks_as_cluster_run == 1).all()), f"share of identical ranks: {INFO.same_ranks_as_cluster_run.mean():.6f}")
check(SEC, "largest difference of a view's top-10 recall from the cluster run",
      0.0, max(abs(top10(RUN[k]["BiPert-ID"]) - top10(REFERENCE.loc[k, "rank_bipert_id"])) for k in RUN), tol=0.001)
show(SEC)

## 1. Full ExPert benchmark (Table 3; intervals in Appendix F)

Means over views of pseudobulk top-10 recall and MRR, by kind of held-out view. *New cell line*: the cell line
was never seen in training. *New donor*: a T-cell donor never seen in training. *Same donor, new condition*: the
donor was seen in training under another condition.

In [ ]:
SEC = "1 full benchmark"
KINDS = ["new cell line", "new donor", "same donor, new condition"]
PAPER_T4 = {"new cell line": (10, 0.426, 0.539, 0.309, 0.415), "new donor": (12, 0.258, 0.381, 0.182, 0.292),
            "same donor, new condition": (12, 0.752, 0.442, 0.625, 0.340), "all views": (34, 0.482, 0.449, 0.376, 0.345)}
PAPER_CI = {  # BiPert-ID minus ExPert: (top-10 difference, low, high), (MRR difference, low, high)
    "new cell line": ((0.113, 0.102, 0.124), (0.107, 0.098, 0.115)),
    "new donor": ((0.123, 0.110, 0.135), (0.109, 0.101, 0.118)),
    "same donor, new condition": ((-0.309, -0.323, -0.296), (-0.285, -0.297, -0.273)),
    "all views": ((-0.033, -0.040, -0.026), (-0.031, -0.037, -0.025))}

def keys_of(kind):
    return [(v.fold, v.dataset) for v in VIEWS.itertuples() if kind == "all views" or v.kind == kind]

if FAST:
    skipped(SEC, "Table 3 and its intervals need all 34 views")
else:
    rows = []
    for kind in KINDS + ["all views"]:
        tabs = [RUN[k] for k in keys_of(kind)]
        rows.append(dict(view=kind, n=len(tabs), top10_ExPert=np.mean([top10(t["ExPert"]) for t in tabs]),
                         top10_BiPertID=np.mean([top10(t["BiPert-ID"]) for t in tabs]),
                         MRR_ExPert=np.mean([mrr(t["ExPert"]) for t in tabs]), MRR_BiPertID=np.mean([mrr(t["BiPert-ID"]) for t in tabs])))
    T4 = pd.DataFrame(rows).set_index("view"); display(T4.round(3))
    for kind, (n, e10, o10, em, om) in PAPER_T4.items():
        check(SEC, f"{kind}: number of views", n, T4.loc[kind, "n"], tol=0)
        check(SEC, f"{kind}: ExPert top-10 recall", e10, T4.loc[kind, "top10_ExPert"])
        check(SEC, f"{kind}: BiPert-ID top-10 recall", o10, T4.loc[kind, "top10_BiPertID"])
        check(SEC, f"{kind}: ExPert MRR", em, T4.loc[kind, "MRR_ExPert"])
        check(SEC, f"{kind}: BiPert-ID MRR", om, T4.loc[kind, "MRR_BiPertID"])
    CI = {kind: paired_bootstrap([RUN[k] for k in keys_of(kind)], "BiPert-ID", "ExPert") for kind in PAPER_CI}
    display(pd.DataFrame({kind: {m: f"{d:+.3f} [{lo:+.3f}, {hi:+.3f}]" for m, (d, lo, hi) in r.items()} for kind, r in CI.items()}).T)
    for kind, paper in PAPER_CI.items():
        for metric, p in zip(("top10", "mrr"), paper):
            for label, pv, rv in zip(("difference", "95% low", "95% high"), p, CI[kind][metric]):
                check(SEC, f"BiPert-ID minus ExPert, {kind}, {metric}: {label}", pv, rv)
    lead = INFO.assign(ahead=INFO.top10 > INFO.top10_ExPert).groupby("kind").ahead.agg(["sum", "size"])
    check_true(SEC, "BiPert-ID leads on every new-donor view", bool(lead.loc["new donor", "sum"] == 12), f"{lead.loc['new donor', 'sum']} of 12")
    check_true(SEC, "BiPert-ID leads on 9 of 10 new cell lines", bool(lead.loc["new cell line", "sum"] == 9), f"{lead.loc['new cell line', 'sum']} of 10")
    check_true(SEC, "ExPert leads on every same-donor, new-condition view", bool(lead.loc["same donor, new condition", "sum"] == 0),
               f"BiPert-ID ahead on {lead.loc['same donor, new condition', 'sum']} of 12")
    # the unweighted version (Appendix B)
    allv = [RUN[k] for k in keys_of("all views")]
    check(SEC, "unweighted version, all views: top-10 recall", 0.424, np.mean([top10(t["unweighted"]) for t in allv]))
    check(SEC, "unweighted version, all views: MRR", 0.324, np.mean([mrr(t["unweighted"]) for t in allv]))
    U = paired_bootstrap(allv, "BiPert-ID", "unweighted")
    for metric, p in (("top10", (0.025, 0.021, 0.029)), ("mrr", (0.021, 0.018, 0.024))):
        for label, pv, rv in zip(("difference", "95% low", "95% high"), p, U[metric]):
            check(SEC, f"BiPert-ID minus unweighted, all views, {metric}: {label}", pv, rv)
show(SEC)

In [ ]:
if not FAST:
    d = INFO.assign(diff=INFO.top10 - INFO.top10_ExPert).sort_values(["kind", "diff"])
    colours = {"new cell line": "#0e7a45", "new donor": "#3a7ca5", "same donor, new condition": "#b0b0b0"}
    fig, ax = plt.subplots(figsize=(7, 6.5))
    ax.barh(range(len(d)), d["diff"], color=[colours[k] for k in d.kind])
    ax.set_yticks(range(len(d))); ax.set_yticklabels([f"{r.dataset} (fold {r.fold})" for r in d.itertuples()], fontsize=6.5)
    ax.axvline(0, color="black", linewidth=0.8); ax.set_xlabel("pseudobulk top-10 recall: BiPert-ID minus ExPert")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colours.values()], labels=list(colours), frameon=False, fontsize=8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_title("All 34 held-out views", loc="left", fontsize=10); plt.tight_layout(); plt.show()

## 2. Four held-out cell lines (Table 2; Table S3)

HepG2, Jurkat, K562 (ExPert's essential-gene screen) and RPE1 are the four lines of the transfer benchmark. The
rows marked † are published values, quoted and not recomputed. ExPert's row is recomputed here from its released
ranks and equals its published values.

In [ ]:
SEC = "2 four cell lines"
LINES = ["HepG2", "Jurkat", "K562", "RPE1"]
LINE_KEY = {v.line: (v.fold, v.dataset) for v in VIEWS.itertuples() if isinstance(v.line, str)}
FOUR = [RUN[LINE_KEY[l]] for l in LINES]
PUBLISHED = {  # (top-10 recall, MRR), mean of the four lines; published by ExPert's authors
    "GEARS†": (0.01, 0.01), "scGPT†": (0.04, 0.02), "GenePert†": (0.06, 0.03), "Own-gene drop only†": (0.18, 0.16),
    "RNA fingerprinting†": (0.22, 0.14), "Centroid†": (0.30, 0.20)}
PAPER_LINE = {"ExPert": ([0.597, 0.503, 0.471, 0.589], [0.457, 0.376, 0.340, 0.434]),
              "BiPert-ID": ([0.688, 0.652, 0.647, 0.637], [0.561, 0.515, 0.520, 0.485])}
per_line = pd.DataFrame({(m, name): [f(t[m]) for t in FOUR] + [np.mean([f(t[m]) for t in FOUR])]
                         for m in ("ExPert", "BiPert-ID") for name, f in (("top-10 recall", top10), ("MRR", mrr))}, index=LINES + ["mean"]).T
display(per_line.round(3))
table3 = pd.DataFrame({**{k: dict(zip(("top-10 recall", "MRR"), v)) for k, v in PUBLISHED.items()},
                       "ExPert": per_line.loc["ExPert", "mean"], "BiPert-ID (ours)": per_line.loc["BiPert-ID", "mean"]}).T
display(table3.round(3))
for m, (p10, pm) in PAPER_LINE.items():
    for l, a, b in zip(LINES, p10, pm):
        check(SEC, f"{m}, {l}: top-10 recall", a, per_line.loc[(m, "top-10 recall"), l])
        check(SEC, f"{m}, {l}: MRR", b, per_line.loc[(m, "MRR"), l])
for m, a, b in (("ExPert", 0.540, 0.402), ("BiPert-ID", 0.656, 0.521)):
    check(SEC, f"{m}, mean of four lines: top-10 recall", a, per_line.loc[(m, "top-10 recall"), "mean"])
    check(SEC, f"{m}, mean of four lines: MRR", b, per_line.loc[(m, "MRR"), "mean"])
B4 = paired_bootstrap(FOUR, "BiPert-ID", "ExPert"); U4 = paired_bootstrap(FOUR, "BiPert-ID", "unweighted")
for name, res, paper in (("BiPert-ID minus ExPert", B4, {"top10": (0.116, 0.099, 0.131), "mrr": (0.119, 0.106, 0.133)}),
                         ("BiPert-ID minus unweighted", U4, {"top10": (0.014, 0.005, 0.021), "mrr": (0.011, 0.005, 0.017)})):
    for metric in ("top10", "mrr"):
        for label, pv, rv in zip(("difference", "95% low", "95% high"), paper[metric], res[metric]):
            check(SEC, f"{name}, four lines, {metric}: {label}", pv, rv)
check_true(SEC, "BiPert-ID outperforms ExPert on every line, both metrics",
           bool((per_line.loc["BiPert-ID", LINES].to_numpy() > per_line.loc["ExPert", LINES].to_numpy()).all()))
show(SEC)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
names = list(table3.index); vals = table3["top-10 recall"].to_numpy(float)
ax.bar(range(len(names)), vals, color=["#c8c8c8"] * (len(names) - 1) + ["#0e7a45"])
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
for i, v in enumerate(vals):
    ax.text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
ax.set_ylabel("pseudobulk top-10 recall"); ax.set_title("Four held-out cell lines (mean)", loc="left", fontsize=10)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout(); plt.show()

## 3. Score ablation (Fig. S2)

The scoring rule has two terms. Each is used alone, then both together. The starred variants read the own-gene
drop only from genes measured in every training cell line (the shared gene space); the full model reads it from
every gene measured in the held-out line, which recovers the candidates whose own gene is outside the shared
space.

In [ ]:
SEC = "3 score ablation"
ABL = {"cosine only": 0.374, "own-gene drop only*": 0.399, "both*": 0.599, "BiPert-ID": 0.656}
abl = pd.DataFrame({name: [top10(t[name]) for t in FOUR] for name in ABL}, index=LINES).T
abl["mean"] = abl.mean(axis=1); display(abl.round(3))
for name, paper in ABL.items():
    check(SEC, f"{name}: top-10 recall, mean of four lines", paper, abl.loc[name, "mean"])
check_true(SEC, "either term alone identifies far fewer perturbations than both", bool(abl.loc[["cosine only", "own-gene drop only*"], "mean"].max() < abl.loc["both*", "mean"] - 0.15))
check_true(SEC, "reading the own-gene drop from every gene of the query line adds a further gain", bool(abl.loc["BiPert-ID", "mean"] > abl.loc["both*", "mean"]))
show(SEC)
fig, ax = plt.subplots(figsize=(5, 2.6))
ax.barh(list(ABL)[::-1], abl["mean"][list(ABL)[::-1]], color=["#0e7a45", "#9ccfb3", "#c8c8c8", "#c8c8c8"])
for i, v in enumerate(abl["mean"][list(ABL)[::-1]]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)
ax.set_xlabel("pseudobulk top-10 recall (mean of four lines)"); ax.set_xlim(0, 0.75)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout(); plt.show()

## 4. Which genes are identified? (Tables 4 and 5)

The knockdowns of each held-out line are cut into quarters of effect size (from the training cell lines), and
separately split by whether the knocked-down gene is one of the 3,096 genes ExPert reads.

In [ ]:
SEC = "4 which genes"
PAIRS = pd.concat([RUN[LINE_KEY[l]].assign(line=l) for l in LINES]).rename_axis("gene").reset_index()
for m in ("ExPert", "BiPert-ID"):
    PAIRS[f"{m} hit"] = (PAIRS[m] < 10).astype(float)
HIT = ["ExPert hit", "BiPert-ID hit"]; QUARTERS = ["Q1 lowest", "Q2", "Q3", "Q4 highest"]

def quarters_within_line(table, column):
    return table.groupby("line")[column].transform(lambda x: pd.qcut(x.rank(method="first"), 4, labels=QUARTERS))

PAIRS["effect_quarter"] = quarters_within_line(PAIRS, "effect_size")
by_effect = PAIRS.groupby("effect_quarter", observed=True)[HIT].mean().loc[QUARTERS]
by_panel = PAIRS.groupby("in_panel")[HIT].mean()
e, o = PAIRS["ExPert hit"] > 0, PAIRS["BiPert-ID hit"] > 0
overlap = {"both": (e & o).mean(), "ExPert alone": (e & ~o).mean(), "BiPert-ID alone": (~e & o).mean(), "neither": (~e & ~o).mean()}
print(f"{len(PAIRS):,d} knockdown-line pairs; in ExPert's panel: {int(PAIRS.in_panel.sum()):,d}, outside: {int((~PAIRS.in_panel).sum()):,d}")
display(by_effect.round(3)); display(by_panel.round(3)); print({k: round(float(v), 3) for k, v in overlap.items()})
for name, pe, po in zip(QUARTERS, (0.21, 0.49, 0.66, 0.79), (0.31, 0.66, 0.77, 0.87)):
    check(SEC, f"effect-size {name}: ExPert", pe, by_effect.loc[name, "ExPert hit"])
    check(SEC, f"effect-size {name}: BiPert-ID", po, by_effect.loc[name, "BiPert-ID hit"])
check(SEC, "in ExPert's panel: ExPert", 0.74, by_panel.loc[True, "ExPert hit"])
check(SEC, "in ExPert's panel: BiPert-ID", 0.80, by_panel.loc[True, "BiPert-ID hit"])
check(SEC, "outside the panel: ExPert", 0.48, by_panel.loc[False, "ExPert hit"])
check(SEC, "outside the panel: BiPert-ID", 0.61, by_panel.loc[False, "BiPert-ID hit"])
check(SEC, "knockdowns in the panel", 756, int(PAIRS.in_panel.sum()), tol=0)
check(SEC, "knockdowns outside the panel", 2584, int((~PAIRS.in_panel).sum()), tol=0)
for name, paper in zip(overlap, (0.48, 0.06, 0.18, 0.29)):
    check(SEC, f"overlap: {name}", paper, overlap[name])
show(SEC)

## 5. Transfer predicts identification (Table 6; Appendix E, Table S2)

The transfer score is computed from the training cell lines only. The outcome is whether the knockdown is
identified in the held-out line (pseudobulk top 10). Because strong effects also transfer better, the logistic
regression predicts the outcome from the standardised transfer score, controlling for the standardised log
effect size, with one intercept per held-out line.

In [ ]:
SEC = "5 transfer predicts identification"
LINK = PAIRS[np.isfinite(PAIRS["transfer"]) & (PAIRS["effect_size"] > 0)].copy()
LINK["log_effect"] = np.log(LINK["effect_size"])
LINK["transfer_quarter"] = quarters_within_line(LINK, "transfer")
by_transfer = LINK.groupby("transfer_quarter", observed=True)[HIT].mean().loc[QUARTERS]
rho = LINK[["transfer", "log_effect"]].corr("spearman").iloc[0, 1]

def regression(table, method):
    y = table[f"{method} hit"].to_numpy(float)
    dummies = np.stack([(table["line"] == l).to_numpy(float) for l in LINES[1:]], 1)
    base = np.column_stack([np.ones(len(table)), z_score(table["log_effect"]), dummies])
    full = np.column_stack([base[:, :1], z_score(table["transfer"]), base[:, 1:]])
    beta, lo, hi, ll = logistic_fit(full, y); _, _, _, ll_base = logistic_fit(base, y)
    p0 = y.mean(); ll0 = (y * np.log(p0) + (1 - y) * np.log(1 - p0)).sum()
    return dict(beta_t=beta[1], low=lo[1], high=hi[1], beta_e=beta[2], r2_without=1 - ll_base / ll0, r2_with=1 - ll / ll0)

strong = LINK[LINK.groupby("line")["log_effect"].transform(lambda x: x >= x.median())]
REG = pd.DataFrame({(subset, m): regression(t, m) for subset, t in (("all", LINK), ("strongest-effect half", strong)) for m in ("ExPert", "BiPert-ID")}).T
print(f"{len(LINK):,d} knockdown-line pairs with a transfer score; strongest-effect half: {len(strong):,d} | "
      f"rank correlation of transfer score and effect size: {rho:.3f}")
display(by_transfer.round(3)); display(REG.round(3))
check(SEC, "number of perturbation-line pairs", 3340, len(LINK), tol=0)
check(SEC, "pairs in the strongest-effect half", 1671, len(strong), tol=0)
for name, pe, po in zip(QUARTERS, (0.19, 0.39, 0.65, 0.90), (0.26, 0.59, 0.82, 0.95)):
    check(SEC, f"transfer {name}: ExPert", pe, by_transfer.loc[name, "ExPert hit"])
    check(SEC, f"transfer {name}: BiPert-ID", po, by_transfer.loc[name, "BiPert-ID hit"])
check(SEC, "rank correlation, transfer score vs effect size", 0.54, rho)
PAPER_REG = {("all", "ExPert"): (1.48, 1.34, 1.62, 0.54, 0.16, 0.29), ("all", "BiPert-ID"): (1.65, 1.48, 1.82, 0.62, 0.17, 0.30),
             ("strongest-effect half", "ExPert"): (1.49, 1.30, 1.68, 0.17, 0.03, 0.22),
             ("strongest-effect half", "BiPert-ID"): (1.41, 1.19, 1.63, 0.35, 0.04, 0.19)}
for key, paper in PAPER_REG.items():
    for col, pv in zip(REG.columns, paper):                       # pseudo-R2 is printed with two decimals in the paper
        check(SEC, f"{key[0]}, {key[1]}: {col}", pv, REG.loc[key, col], tol=0.0075 if col.startswith("r2") else TOL)
show(SEC)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.2))
for column, colour, label in (("ExPert hit", "#8a8a8a", "ExPert"), ("BiPert-ID hit", "#0e7a45", "BiPert-ID (ours)")):
    ax.plot(QUARTERS, by_transfer[column], color=colour, linewidth=2, marker="o", markersize=6, label=label)
ax.set_ylim(0, 1); ax.set_ylabel("fraction identified (pseudobulk top 10)"); ax.set_xlabel("quarter of the transfer score (training cell lines only)")
ax.set_title("Perturbations whose effect transfers are the ones identified", loc="left", fontsize=10)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()

## Summary: every check

In [ ]:
SUMMARY = pd.DataFrame(CHECKS)
counts = SUMMARY["OK"].value_counts()
print(f"{counts.get('PASS', 0)} of {len(SUMMARY) - SUMMARY['OK'].str.startswith('SKIPPED').sum()} checks pass"
      + (f"; {SUMMARY['OK'].str.startswith('SKIPPED').sum()} skipped (FAST)" if FAST else "")
      + f" | runtime {(time.time() - T_START) / 60:.1f} minutes")
display(SUMMARY[SUMMARY["OK"] == "FAIL"] if (SUMMARY["OK"] == "FAIL").any() else "no failed check")
display(SUMMARY)